In [1]:
!pip install transformers torch onnx onnxruntime optimum onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import json

In [3]:
# Step 1: Define the model architecture
class TinyBERTDualClassifier(nn.Module):
    def __init__(self, num_module_labels, num_date_labels, dropout_rate=0.1):
        super(TinyBERTDualClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained("JayShah07/tinybert-dual-classifier")
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(p=dropout_rate)
        self.module_classifier = nn.Linear(self.hidden_size, num_module_labels)
        self.date_classifier = nn.Linear(self.hidden_size, num_date_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        module_logits = self.module_classifier(cls_output)
        date_logits = self.date_classifier(cls_output)
        return module_logits, date_logits, cls_output

# Step 2: Load the model weights
print("Loading model...")
classifier_config = torch.hub.load_state_dict_from_url(
    "https://huggingface.co/JayShah07/tinybert-dual-classifier/resolve/main/classifier_heads.pt",
    map_location=torch.device('cpu')
)

Loading model...
Downloading: "https://huggingface.co/JayShah07/tinybert-dual-classifier/resolve/main/classifier_heads.pt" to /root/.cache/torch/hub/checkpoints/classifier_heads.pt


100%|██████████| 18.8k/18.8k [00:00<00:00, 407kB/s]


In [5]:
model = TinyBERTDualClassifier(
    num_module_labels=6,
    num_date_labels=7,
    dropout_rate=0.0  # Set to 0 for inference
)

model.module_classifier.load_state_dict(classifier_config['module_classifier'])
model.date_classifier.load_state_dict(classifier_config['date_classifier'])
model.eval()

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("JayShah07/tinybert-dual-classifier")

model.safetensors:   0%|          | 0.00/57.4M [00:00<?, ?B/s]

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [6]:
print("Exporting to ONNX...")

# Create dummy inputs
dummy_text = "Show my holdings for this month"
dummy_inputs = tokenizer(
    dummy_text,
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=128
)

input_ids = dummy_inputs['input_ids']
attention_mask = dummy_inputs['attention_mask']

# 🔥 IMPORTANT FIXES ARE BELOW
torch.onnx.export(
    model,
    (input_ids, attention_mask),
    "tinybert_dual_classifier.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,

    # ----------------------------
    # INPUT NAMES (unchanged)
    # ----------------------------
    input_names=['input_ids', 'attention_mask'],

    # 🔥 FIX 1: ADD cls_embedding OUTPUT
    output_names=[
        'module_logits',
        'date_logits',
        'cls_embedding'  # <-- THIS FIXES EMBEDDING EXPORT
    ],

    # 🔥 FIX 2: ADD dynamic axis FOR EMBEDDING
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'module_logits': {0: 'batch_size'},
        'date_logits': {0: 'batch_size'},
        'cls_embedding': {0: 'batch_size'}  # <-- REQUIRED
    },

    # This is fine to keep
    keep_initializers_as_inputs=True
)

print("ONNX model exported successfully!")

Exporting to ONNX...


/tmp/ipython-input-3789816317.py:17: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1230 14:47:39.429000 2023 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `TinyBERTDualClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyBERTDualClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅
Applied 21 of general pattern rewrite rules.
ONNX model exported successfully!


In [7]:
print("Quantizing model for faster inference...")
quantize_dynamic(
    "tinybert_dual_classifier.onnx",
    "tinybert_dual_classifier_quantized.onnx",
    weight_type=QuantType.QUInt8
)
print("Quantized model created!")

# ==================== SAVE TOKENIZER VOCABULARY ====================
print("Saving tokenizer data...")

# Save vocabulary
vocab = tokenizer.get_vocab()
with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

# Save tokenizer config
tokenizer_config = {
    'max_length': 128,
    'padding': 'max_length',
    'truncation': True,
    'vocab_size': len(vocab),
    'cls_token': tokenizer.cls_token,
    'sep_token': tokenizer.sep_token,
    'pad_token': tokenizer.pad_token,
    'unk_token': tokenizer.unk_token,
    'cls_token_id': tokenizer.cls_token_id,
    'sep_token_id': tokenizer.sep_token_id,
    'pad_token_id': tokenizer.pad_token_id,
    'unk_token_id': tokenizer.unk_token_id,
}

with open('tokenizer_config.json', 'w') as f:
    json.dump(tokenizer_config, f, indent=2)

# Save label mappings
labels_config = {
    'module_labels': ['holdings', 'capital_gains', 'scheme_wise_returns',
                      'investment_account_wise_returns', 'portfolio_update', 'None_module'],
    'date_labels': ['Current Year', 'Previous Year', 'Daily', 'Monthly',
                    'Weekly', 'Yearly', 'None_date']
}

with open('labels.json', 'w') as f:
    json.dump(labels_config, f, indent=2)


Quantizing model for faster inference...


Quantized model created!
Saving tokenizer data...


In [8]:
print("\nVerifying ONNX model...")
import onnxruntime as ort

# Test original model
ort_session = ort.InferenceSession("tinybert_dual_classifier.onnx")
ort_inputs = {
    'input_ids': input_ids.numpy(),
    'attention_mask': attention_mask.numpy()
}
ort_outputs = ort_session.run(None, ort_inputs)

print(f"Module prediction: {ort_outputs[0].argmax()}")
print(f"Date prediction: {ort_outputs[1].argmax()}")
print(f"Embeddings:{ort_outputs[2]}")

# Test quantized model
ort_session_q = ort.InferenceSession("tinybert_dual_classifier_quantized.onnx")
ort_outputs_q = ort_session_q.run(None, ort_inputs)

print(f"\nQuantized - Module prediction: {ort_outputs_q[0].argmax()}")
print(f"Quantized - Date prediction: {ort_outputs_q[1].argmax()}")



Verifying ONNX model...
Module prediction: 0
Date prediction: 3
Embeddings:[[ 0.01367881  0.51729447 -0.63619393 -0.06929214 -0.68677926  0.18771301
  -0.80643433  0.45956382  0.79840726  0.4608375   0.16952866 -0.08243541
  -0.4759381  -0.09534253  0.9874371   0.638468   -0.48877776 -0.43974188
  -0.74662846 -0.53356224  0.43847936  0.01671754 -0.5086323   0.18188372
  -0.05182113 -0.27945825  0.75896657  0.5693373  -0.22214797  0.62115043
   0.23487845 -0.13684143  0.78242636 -0.25415948 -0.8109382  -0.41418323
   0.42363292 -0.39369494 -0.85025185 -0.10709454 -0.01641084 -0.5553512
   1.1305444   0.19319506 -0.5585646  -0.02224254  1.0293059   0.31687912
  -0.5057528  -0.11692786 -0.8007085  -0.12842198  0.08357018  0.20038193
  -0.21249525 -0.17517722  0.2658754   0.64522946 -0.23114716  0.48114485
   0.1095956  -0.26750642 -0.5046635   0.4721357   0.47727016 -0.4045631
  -0.45072243  0.68596685 -0.64758694  0.40917924 -0.45037362  0.41420585
   0.31674117  0.9273688   0.46591577 

In [9]:
import os
print("\n" + "="*50)
print("FILE SIZES:")
print("="*50)
original_size = os.path.getsize("tinybert_dual_classifier.onnx") / (1024*1024)
quantized_size = os.path.getsize("tinybert_dual_classifier_quantized.onnx") / (1024*1024)
print(f"Original ONNX model: {original_size:.2f} MB")
print(f"Quantized ONNX model: {quantized_size:.2f} MB")
print(f"Size reduction: {((original_size - quantized_size) / original_size * 100):.1f}%")

print("\n" + "="*50)
print("DOWNLOAD THESE FILES:")
print("="*50)
print("1. tinybert_dual_classifier_quantized.onnx (recommended)")
print("2. vocab.json")
print("3. tokenizer_config.json")
print("4. labels.json")
print("\nOptional:")
print("5. tinybert_dual_classifier.onnx (if you want non-quantized version)")
print("="*50)


FILE SIZES:
Original ONNX model: 0.04 MB
Quantized ONNX model: 13.69 MB
Size reduction: -38846.5%

DOWNLOAD THESE FILES:
1. tinybert_dual_classifier_quantized.onnx (recommended)
2. vocab.json
3. tokenizer_config.json
4. labels.json

Optional:
5. tinybert_dual_classifier.onnx (if you want non-quantized version)
